# Revenue Forecast Verification
Linear Regression on `orders.csv` — used to verify the agent's forecast output.

This notebook replicates exactly what `src/forecaster.py` does:
1. Load orders.csv
2. Aggregate to monthly revenue
3. Fit LinearRegression on integer time index
4. Predict next 3 months

In [2]:
!pip install matplotlib scikit-learn

  Using cached matplotlib-3.11.0-cp313-cp313-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached matplotlib-3.11.0-cp313-cp313-win_amd64.whl (9.3 MB)
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)

   ------------- -------------------------- 1/3 [contourpy]
   ------------- -------------------------- 1/3 [contourpy]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matplotlib]



ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'D:\\Samyak\\FullStack_Projects\\Agentic_Analyst\\venv\\Lib\\site-packages\\matplotlib\\_tight_layout.py'
Check the permissions.



In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

Matplotlib is building the font cache; this may take a moment.


   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.3 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.3 MB 5.9 MB/s eta 0:00:02
   ----------- ---------------------------- 2.6/9.3 MB 6.7 MB/s eta 0:00:01
   ------------ --------------------------- 2.9/9.3 MB 4.1 MB/s eta 0:00:02
   ------------------- -------------------- 4.5/9.3 MB 5.0 MB/s eta 0:00:01
   ------------------------------- -------- 7.3/9.3 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 6.9 MB/s  0:00:01
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ----------------------------------- ---- 2.1/2.3 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 8.8 MB/s  0:00:00

   ---------------------------------------- 0/6 [pyparsing]
   ---------------------------------------- 0/6 [pypa

## Step 1 — Load and inspect orders.csv

In [4]:
df = pd.read_csv('./data/orders.csv')
df['order_date'] = pd.to_datetime(df['order_date'])

print(f'Total rows : {len(df):,}')
print(f'Date range : {df["order_date"].min().date()} → {df["order_date"].max().date()}')
print(f'Columns    : {list(df.columns)}')
df.head()

Total rows : 50,000
Date range : 2024-06-08 → 2026-06-05
Columns    : ['order_id', 'user_id', 'product_id', 'order_date', 'quantity', 'total_amount']


C:\Users\Samyak\AppData\Local\Temp\ipykernel_28180\362030042.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['order_date'] = pd.to_datetime(df['order_date'])


,order_id,user_id,product_id,order_date,quantity,total_amount
0,39,67,21,2026-04-30,5,2499.95
1,95,10,21,2026-05-15,5,2499.95
2,427,44,21,2025-11-26,5,2499.95
3,1280,9,21,2026-04-10,5,2499.95
4,1822,3,21,2025-08-07,5,2499.95


## Step 2 — Aggregate to monthly revenue
This replicates the SQL the agent generates:
```sql
SELECT DATE_TRUNC('month', order_date) AS forecast_date,
       SUM(total_amount) AS forecast_value
FROM orders
GROUP BY forecast_date
ORDER BY forecast_date ASC;
```

In [5]:
monthly = (
    df.groupby(df['order_date'].dt.to_period('M'))['total_amount']
    .sum()
    .reset_index()
    .rename(columns={'order_date': 'month', 'total_amount': 'revenue'})
)
monthly['month_str'] = monthly['month'].astype(str)
monthly['index']     = range(len(monthly))  # integer time index for regression

print(f'Monthly data points: {len(monthly)}')
print()
monthly[['month_str', 'revenue']]

Monthly data points: 25



,month_str,revenue
0,2024-06,11607.17
1,2024-07,43847.65
2,2024-08,76878.43
3,2024-09,149810.04
4,2024-10,216437.04
5,2024-11,226479.84
6,2024-12,290102.06
7,2025-01,318309.98
8,2025-02,342609.14
9,2025-03,419773.58


## Step 3 — Fit Linear Regression
X = integer time index [0, 1, 2, ..., N-1]

y = monthly revenue

This is exactly what `_run_linear_regression()` in `src/forecaster.py` does.

In [6]:
X = monthly[['index']].values
y = monthly['revenue'].values

model = LinearRegression()
model.fit(X, y)

print(f'Slope (revenue increase per month) : ${model.coef_[0]:,.2f}')
print(f'Intercept (baseline revenue)       : ${model.intercept_:,.2f}')
print(f'R² score (fit quality, 1.0 = perfect): {model.score(X, y):.4f}')

Slope (revenue increase per month) : $71,039.12
Intercept (baseline revenue)       : $-147,890.92
R² score (fit quality, 1.0 = perfect): 0.7215


## Step 4 — Predict next 3 months
These are the values the agent should return.

In [7]:
n        = len(monthly)
future_X = np.array([[n], [n+1], [n+2]])
preds    = model.predict(future_X)

# Generate future month labels
last_period   = monthly['month'].iloc[-1]
future_months = [str(last_period + i) for i in range(1, 4)]

print('=== FORECAST — Next 3 Months ===')
print()
for month, val in zip(future_months, preds):
    print(f'  {month}   ${val:>14,.2f}')

print()
print('Compare these values against what the agent returns.')

=== FORECAST — Next 3 Months ===

  2026-07   $  1,628,087.11
  2026-08   $  1,699,126.23
  2026-09   $  1,770,165.35

Compare these values against what the agent returns.


## Step 5 — Plot historical + forecast

In [ ]:
# Historical fitted line
fitted = model.predict(X)

fig, ax = plt.subplots(figsize=(14, 5))

# Actual monthly revenue
ax.bar(monthly['index'], monthly['revenue'],
       color='steelblue', alpha=0.6, label='Actual monthly revenue')

# Fitted trend line
ax.plot(monthly['index'], fitted,
        color='navy', linewidth=2, linestyle='--', label='Linear trend (fitted)')

# Forecast points
future_idx = [n, n+1, n+2]
ax.scatter(future_idx, preds,
           color='red', s=100, zorder=5, label='Forecast (next 3 months)')
ax.plot([n-1, n, n+1, n+2],
        [fitted[-1]] + list(preds),
        color='red', linewidth=1.5, linestyle='--')

# Annotate forecast values
for idx, month, val in zip(future_idx, future_months, preds):
    ax.annotate(f'{month}\n${val:,.0f}',
                xy=(idx, val), xytext=(0, 12),
                textcoords='offset points', ha='center', fontsize=8, color='red')

# X axis labels — show every 3rd month
all_labels = list(monthly['month_str']) + future_months
all_idx    = list(monthly['index']) + future_idx
ax.set_xticks(all_idx[::3])
ax.set_xticklabels(all_labels[::3], rotation=45, ha='right', fontsize=8)

ax.set_title('Monthly Revenue — Historical + 3-Month Linear Regression Forecast',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6 — Exclude incomplete current month (optional)
June 2026 only has partial data (mid-month cutoff).
Rerun the forecast excluding it to see if predictions change significantly.

In [ ]:
# Drop the last row (incomplete current month)
monthly_clean = monthly.iloc[:-1].copy()
monthly_clean['index'] = range(len(monthly_clean))

X2 = monthly_clean[['index']].values
y2 = monthly_clean['revenue'].values

model2 = LinearRegression()
model2.fit(X2, y2)

n2           = len(monthly_clean)
future_X2    = np.array([[n2], [n2+1], [n2+2]])
preds2       = model2.predict(future_X2)
last_period2 = monthly_clean['month'].iloc[-1]
months2      = [str(last_period2 + i) for i in range(1, 4)]

print('=== FORECAST (excluding incomplete June 2026) ===')
print()
for month, val in zip(months2, preds2):
    print(f'  {month}   ${val:>14,.2f}')

print()
print(f'R² with full data    : {model.score(X, y):.4f}')
print(f'R² without last month: {model2.score(X2, y2):.4f}')